# Deep Learning Lab 0
In this lab we will be training Convolutional Neural Networks and Visualize the progress through Wandb.
# Task 0.1

## Imports

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import random_split

import torchvision.models as models
from torchvision.models import alexnet, AlexNet_Weights

import wandb
import os



/usr/local/lib/python3.11/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


## Wandb Setup


In [2]:
# Storage of wandb legend data

xs = list(range(500))
ys_train = []
ys_valid = []

## Preprocessing
### Checklist Train Data
- Normalization
- Data Augmentation
- Create images --> ToTensor()
- Dataloader
### Checklist Test Data
- Normalization
- Create images --> ToTensor()

In [3]:
"""
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Randomly Crops 32x32 Region After Padding To Create Small Shifts --> Robustness
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

"""

transformer = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

transformer_MNIST = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5), std=(0.5))
    ])

transformer_SVHN = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5), std=(0.5))
])

traindata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transformer)

testdata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=transformer)

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
train_size = int(0.8*len(traindata))
val_size = len(traindata) - train_size

train_dataset, val_dataset = random_split(traindata, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload = torch.utils.data.DataLoader(traindata, batch_size=32, shuffle=True)
valload = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
testload = torch.utils.data.DataLoader(testdata, batch_size=32, shuffle=False)

## Training and Testing 
### Checklist
- Activation Function LeakyReLU
- Optimizer Stochastic Gradient Descent lr = 0.0001
- Accuracy on Test set
- Val/Train Stopping when the Val loss converges
- Tensorboard and wandb
- Schedulers


In [5]:
if torch.cuda.is_available():
    device = "cuda" #Nvidia Graphics Card
elif torch.backends.mps.is_available():
    device = "mps" # Apple
else:
    device = "cpu" # Worst Case
print(device)


cuda


In [6]:
class CNN_LReLU(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_LReLU, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class CNN_MNIST(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_MNIST, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
model = CNN_LReLU(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
num_epochs = 500
early_stop_patience = 6
early_stopping_counter = 0

In [8]:
# ------------------------- Wandb -------------------------
wandb.init(project="DeepLearning", name="CIFAR10_SGD_LeakyReLU")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    ys_train.append(average_loss)
    
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

            

        average_val_loss = val_running_loss / len(valload)
        ys_valid.append(average_val_loss)
    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        early_stopping_counter = 0  
        torch.save(model.state_dict(), "BestModel_SGDReLu.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter:{early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.8f}"
    )

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

wandb.log({
    "Loss - CIFAR10_SGD_LeakyReLU": wandb.plot.line_series(
        xs=list(range(500)),
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})    
wandb.finish()
ys_train = []
ys_valid = []

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: david_samf (joanna-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Saved The Best Performing Model
Epoch [1/500] train loss: 2.174, val loss: 2.054, lr: 0.00010000
Saved The Best Performing Model
Epoch [2/500] train loss: 1.987, val loss: 1.900, lr: 0.00010000
Saved The Best Performing Model
Epoch [3/500] train loss: 1.866, val loss: 1.794, lr: 0.00010000
Saved The Best Performing Model
Epoch [4/500] train loss: 1.782, val loss: 1.718, lr: 0.00010000
Saved The Best Performing Model
Epoch [5/500] train loss: 1.719, val loss: 1.656, lr: 0.00010000
Saved The Best Performing Model
Epoch [6/500] train loss: 1.668, val loss: 1.611, lr: 0.00010000
Saved The Best Performing Model
Epoch [7/500] train loss: 1.628, val loss: 1.572, lr: 0.00010000
Saved The Best Performing Model
Epoch [8/500] train loss: 1.592, val loss: 1.534, lr: 0.00010000
Saved The Best Performing Model
Epoch [9/500] train loss: 1.561, val loss: 1.503, lr: 0.00010000
Saved The Best Performing Model
Epoch [10/500] train loss: 1.535, val loss: 1.475, lr: 0.00010000
Saved The Best Performing Mod

In [9]:
model.load_state_dict(torch.load("BestModel.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()        

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



Test loss: 1.509
Test accuracy: 48.76%


### Checklist 2
- LeakyReLU --> Tanh
- Optimizer: SGD --> Adam
- Visualize the results on a Tensorboard

In [10]:
class CNN_tanh(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_tanh, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.Tanh(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.Tanh(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [11]:
# the first model with LeakyReLU activation function and Adam optimizer
model_LReLU = CNN_LReLU(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epochs = num_epochs
early_stop_patience = 6
early_stopping_counter = 0

In [12]:
wandb.init(project="DeepLearning", name="CIFAR10_Adam_LeakyReLU")

num_epochs = num_epochs

# Initialize loss tracking lists
ys_train = []
ys_valid = []

# ------------------------- Wandb -------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf') 

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    ys_train.append(average_loss)
    
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload)
        ys_valid.append(average_val_loss)
        
    scheduler.step(average_loss)
    
    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_Task_AdamLReLU.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter:{early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

    # Early Stopping
    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

# Log combined plot at end of training
wandb.log({
    "Loss - CIFAR10 Learning with Adam + ReLU and Adam + Tanh": wandb.plot.line_series(
        xs=list(range(num_epochs)),
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})
wandb.finish()
ys_train = []
ys_valid = []

Saved The Best Performing Model
Epoch [1/500] train loss: 1.282, val loss: 1.005, lr: 0.000100
Saved The Best Performing Model
Epoch [2/500] train loss: 1.030, val loss: 0.886, lr: 0.000100
Saved The Best Performing Model
Epoch [3/500] train loss: 0.916, val loss: 0.790, lr: 0.000100
Saved The Best Performing Model
Epoch [4/500] train loss: 0.839, val loss: 0.744, lr: 0.000100
Saved The Best Performing Model
Epoch [5/500] train loss: 0.778, val loss: 0.686, lr: 0.000100
Saved The Best Performing Model
Epoch [6/500] train loss: 0.722, val loss: 0.615, lr: 0.000100
Saved The Best Performing Model
Epoch [7/500] train loss: 0.669, val loss: 0.554, lr: 0.000100
Saved The Best Performing Model
Epoch [8/500] train loss: 0.624, val loss: 0.525, lr: 0.000100
Saved The Best Performing Model
Epoch [9/500] train loss: 0.581, val loss: 0.477, lr: 0.000100
Saved The Best Performing Model
Epoch [10/500] train loss: 0.542, val loss: 0.432, lr: 0.000100
Saved The Best Performing Model
Epoch [11/500] tr

In [13]:
# the first (CNN) model with tanh activation function and Adam optimizer
model_tanh = CNN_tanh(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # 
num_epochs = num_epochs
early_stop_patience = 6
early_stopping_counter = 0

In [14]:
# ------------------------- Wandb -------------------------
wandb.init(project="DeepLearning", name="CIFAR10 Adam tanh")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')
    

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    ys_train.append(average_loss)
    
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload)
        ys_valid.append(average_val_loss)
        
    scheduler.step(average_loss)

    # ------------------------- Saving Best Model and EarlyStopping-------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_Task_AdamTanh.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter:{early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

    # Early Stopping
    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

# Log combined plot at end of training
wandb.log({
    "Loss - CIFAR10 Learning with Adam + ReLU and Adam + Tanh": wandb.plot.line_series(
        xs=list(range(num_epochs)),
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})
wandb.finish()
ys_train = []
ys_valid = []

Saved The Best Performing Model
Epoch [1/500] train loss: 0.094, val loss: 0.034, lr: 0.000100
Saved The Best Performing Model
Epoch [2/500] train loss: 0.087, val loss: 0.032, lr: 0.000100
No improvement, increasing counter:1
Epoch [3/500] train loss: 0.081, val loss: 0.035, lr: 0.000100
No improvement, increasing counter:2
Epoch [4/500] train loss: 0.079, val loss: 0.044, lr: 0.000100
No improvement, increasing counter:3
Epoch [5/500] train loss: 0.076, val loss: 0.034, lr: 0.000100
Saved The Best Performing Model
Epoch [6/500] train loss: 0.074, val loss: 0.023, lr: 0.000100
Saved The Best Performing Model
Epoch [7/500] train loss: 0.069, val loss: 0.021, lr: 0.000100
No improvement, increasing counter:4
Epoch [8/500] train loss: 0.065, val loss: 0.023, lr: 0.000100
No improvement, increasing counter:5
Epoch [9/500] train loss: 0.063, val loss: 0.030, lr: 0.000100
Saved The Best Performing Model
Epoch [10/500] train loss: 0.064, val loss: 0.019, lr: 0.000100
Saved The Best Performin

In [15]:
model.load_state_dict(torch.load("BestModel_Task_AdamTanh.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()         

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



Test loss: 1.267
Test accuracy: 72.89%


# Task 0.2

## 0.2 Task
### 0.2.1 Transfer Learning from ImageNet
- Download and prepare CIFAR-10 dataset (it is already available in the above mentioned libraries)
- Use AlexNet as the model (Pytorch AlexNet)
- You have to perform two separate experiments-
    - Train the model for CIFAR-10 data, Report the test test accuracy. (also referred as fine tuning the model)
    - Use the pretarined weights of AlexNet, in other words use AlexNet as a pretrained network for image classification on CIFAR-10 data (also referred as Feature Extraction), Report the test test accuracy. (optional)
- In both the above cases remember to add an extra fully connected layer to the classifier with number of neurons = 10, because there are 10 classes in CIFAR-10 dataset. This layer will be trainable in both cases.
- Explain (briefly!) what is the difference between the two runs and why there is a difference in performance. (optional)

In [16]:
early_stop_patience = 6
early_stopping_counter = 0
epochs = 500
learning_rate = 0.001

In [17]:
wandb.init(project="DeepLearning", name="CIFAR10_FineTuning_AlexNet")

# Initialize loss tracking lists
ys_train = []
ys_valid = []

train_transformer = transforms.Compose([
    transforms.Resize((256)),
    transforms.RandomCrop((227)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])  

val_test_transformer = transforms.Compose([
    transforms.Resize((227)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

# Download CIFAR-10 datasets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transformer)
valset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=val_test_transformer)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_test_transformer)


train_size = int(0.8 * len(trainset))
val_size = len(trainset) - train_size

train_dataset, _ = random_split(trainset, [train_size, val_size], 
                                 generator=torch.Generator().manual_seed(42))
_, val_dataset = random_split(valset, [train_size, val_size], 
                              generator=torch.Generator().manual_seed(42))


trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
valloader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

print(f"Dataset Split - Train: {train_size}, Val: {val_size}, Test: {len(testset)}")


print("\n" + "=" * 50)
print("EXPERIMENT 1: Fine-Tuning Pretrained AlexNet")
print("=" * 50)

model_finetune = models.alexnet(pretrained=True)
# Replace the last fully connected layer to output 10 classes (CIFAR-10)
model_finetune.classifier[6] = nn.Linear(4096, 10)
model_finetune = model_finetune.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_finetune = torch.optim.Adam(model_finetune.parameters(), lr=learning_rate)

# Training loop for fine-tuning
best_val_loss = float('inf')
for epoch in range(epochs):
    model_finetune.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_finetune.zero_grad()
        
        outputs = model_finetune(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_finetune.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    ys_train.append(avg_train_loss)

    # Validation
    model_finetune.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_finetune(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valloader)
    ys_valid.append(avg_val_loss)
    
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model_finetune.state_dict(), "BestModel_Finetune.pth")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter: {early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break
# Test the fine-tuned model
model_finetune.load_state_dict(torch.load("BestModel_Finetune.pth"))
model_finetune.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_finetune(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_finetune = 100 * (correct / total)
print(f"\nTest Accuracy (Fine-tuning): {accuracy_finetune:.2f}%")

# Log combined plot at end of training
wandb.log({
    "Loss - AlexNet": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})
wandb.finish()
ys_train = []
ys_valid = []

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Dataset Split - Train: 40000, Val: 10000, Test: 10000

EXPERIMENT 1: Fine-Tuning Pretrained AlexNet


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/500] Train Loss: 1.8425 | Val Loss: 1.4956
Epoch [2/500] Train Loss: 1.3962 | Val Loss: 1.2164
Epoch [3/500] Train Loss: 1.1692 | Val Loss: 1.0000
Epoch [4/500] Train Loss: 1.0383 | Val Loss: 0.9061
Epoch [5/500] Train Loss: 0.9739 | Val Loss: 0.8820
Epoch [6/500] Train Loss: 0.9150 | Val Loss: 0.8471
Epoch [7/500] Train Loss: 0.8801 | Val Loss: 0.7651
Epoch [8/500] Train Loss: 0.8332 | Val Loss: 0.7801
No improvement, increasing counter: 1
Epoch [10/500] Train Loss: 0.7872 | Val Loss: 0.7562
Epoch [11/500] Train Loss: 0.7725 | Val Loss: 0.7134
Epoch [12/500] Train Loss: 0.7566 | Val Loss: 0.7684
No improvement, increasing counter: 3
Epoch [13/500] Train Loss: 0.7430 | Val Loss: 0.7101
Epoch [14/500] Train Loss: 0.7203 | Val Loss: 0.7228
No improvement, increasing counter: 4
Epoch [15/500] Train Loss: 0.7169 | Val Loss: 0.7012
Epoch [16/500] Train Loss: 0.6843 | Val Loss: 0.6911
Epoch [17/500] Train Loss: 0.7018 | Val Loss: 0.6738
Epoch [18/500] Train Loss: 0.6786 | Val Loss: 

In [18]:
early_stop_patience = 6
early_stopping_counter = 0
epochs = 500
learning_rate = 0.001

In [19]:
# ======================== EXPERIMENT 2: Feature Extraction (Frozen Pretrained AlexNet) ========================
wandb.init(project="DeepLearning", name="CIFAR10_FeatureExtraction_AlexNet")

print("\n" + "=" * 50)
print("EXPERIMENT 2: Feature Extraction (Frozen AlexNet)")
print("=" * 50)

model_feature_extract = models.alexnet(pretrained=True)

# Freeze all layers except the final classifier layer
for param in model_feature_extract.features.parameters():
    param.requires_grad = False

# Only the newly added layer will be trainable
model_feature_extract.classifier[6] = nn.Linear(4096, 10)
model_feature_extract = model_feature_extract.to(device)

# Only optimize the new classifier layer
optimizer_feature_extract = torch.optim.Adam(model_feature_extract.classifier[6].parameters(), lr=learning_rate)

# Training loop for feature extraction
best_val_loss_fe = float('inf')
for epoch in range(epochs):
    model_feature_extract.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_feature_extract.zero_grad()
        
        outputs = model_feature_extract(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_feature_extract.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    ys_train.append(avg_train_loss)

    # Validation
    model_feature_extract.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_feature_extract(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valloader)
    
    ys_valid.append(avg_val_loss)
    
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss_fe:
        best_val_loss_fe = avg_val_loss
        torch.save(model_feature_extract.state_dict(), "BestModel_FeatureExtract.pth")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter: {early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

# Test the feature extraction model
model_feature_extract.load_state_dict(torch.load("BestModel_FeatureExtract.pth"))
model_feature_extract.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_feature_extract(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_feature_extract = 100 * (correct / total)
print(f"\nTest Accuracy (Feature Extraction): {accuracy_feature_extract:.2f}%")
wandb.log({"feature_extract_test_accuracy": accuracy_feature_extract})

# ======================== COMPARISON ========================
print("\n" + "=" * 50)
print("COMPARISON SUMMARY")
print("=" * 50)
print(f"Fine-Tuning Test Accuracy:      {accuracy_finetune:.2f}%")
print(f"Feature Extraction Accuracy:    {accuracy_feature_extract:.2f}%")
print(f"Difference:                     {abs(accuracy_finetune - accuracy_feature_extract):.2f}%")
print("\nExplanation:")
print("- Fine-tuning allows all layers to be updated with CIFAR-10 data,")
print("  adapting the learned ImageNet features to the new task.")
print("- Feature extraction keeps the pre-trained weights frozen,")
print("  using ImageNet features as fixed representations.")
print("- Fine-tuning typically performs better as it can specialize")
print("  the model filters to the specific CIFAR-10 task.")

# Log combined plot
wandb.log({
    "Loss - AlexNet": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})
wandb.finish()
ys_train = []
ys_valid = []


EXPERIMENT 2: Feature Extraction (Frozen AlexNet)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/500] Train Loss: 1.0076 | Val Loss: 0.7003
Epoch [2/500] Train Loss: 0.8934 | Val Loss: 0.7293
No improvement, increasing counter: 1
Epoch [3/500] Train Loss: 0.8675 | Val Loss: 0.6731
Epoch [4/500] Train Loss: 0.8558 | Val Loss: 0.6804
No improvement, increasing counter: 2
Epoch [5/500] Train Loss: 0.8510 | Val Loss: 0.6807
No improvement, increasing counter: 3
Epoch [6/500] Train Loss: 0.8501 | Val Loss: 0.6752
No improvement, increasing counter: 4
Epoch [7/500] Train Loss: 0.8476 | Val Loss: 0.7317
No improvement, increasing counter: 5
Epoch [8/500] Train Loss: 0.8454 | Val Loss: 0.6930
No improvement, increasing counter: 6
Epoch [9/500] Train Loss: 0.8370 | Val Loss: 0.6741
No improvement, increasing counter: 7
Early stop initiated

Test Accuracy (Feature Extraction): 75.36%

COMPARISON SUMMARY
Fine-Tuning Test Accuracy:      78.75%
Feature Extraction Accuracy:    75.36%
Difference:                     3.39%

Explanation:
- Fine-tuning allows all layers to be updated with 

feature_extract_test_accuracy,▁
feature_extract_test_accuracy,75.36


### Task 0.2.2

Prepare a CNN of your choice and train it on the MNIST data. Report the accuracy

In [20]:
#Task 2.2
traindata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transformer_MNIST)

testdata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transformer_MNIST)


train_size = int(0.8*len(traindata_MNIST))
val_size = len(traindata_MNIST) - train_size

train_dataset_MNIST, val_dataset_MNIST = random_split(traindata_MNIST, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_MNIST = torch.utils.data.DataLoader(traindata_MNIST, batch_size=32, shuffle=True)
valload_MNIST = torch.utils.data.DataLoader(val_dataset_MNIST, batch_size=32, shuffle=False)
testload_MNIST = torch.utils.data.DataLoader(testdata_MNIST, batch_size=32, shuffle=False)

model = CNN_MNIST(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
num_epochs = 10

In [21]:
early_stop_patience = 6
early_stopping_counter = 0
epochs = 500

In [22]:
wandb.init(project="DeepLearning", name="MNIST Training with CNN")

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_MNIST)
    ys_train.append(average_loss)
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_MNIST:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_MNIST)

        ys_valid.append(average_val_loss)

    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStop -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_MNIST.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter: {early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break
    
# Log combined plot
wandb.log({
    "Loss - MNIST Learning with CNN": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})
wandb.finish()
ys_train = []
ys_valid = []

Saved The Best Performing Model
Epoch [1/10] train loss: 1.830, val loss: 1.372, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 1.095, val loss: 0.852, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 0.738, val loss: 0.611, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 0.563, val loss: 0.485, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 0.462, val loss: 0.402, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 0.394, val loss: 0.348, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 0.348, val loss: 0.312, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 0.313, val loss: 0.281, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 0.287, val loss: 0.257, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 0.264, val loss: 0.239, lr: 0.000100


In [23]:
#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()        

average_test_loss = test_loss / len(testload_MNIST)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")


Test loss: 0.219
Test accuracy: 95.00%


Use the above model as a pre-trained CNN for the SVHN dataset. Report the accuracy

In [24]:
#Import SVHN Dataset 0.2.2
traindata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="train",
    download=True,
    transform=transformer_SVHN)

testdata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="test",
    download=True,
    transform=transformer_SVHN)


train_size = int(0.8*len(traindata_SVHN))
val_size = len(traindata_SVHN) - train_size

train_dataset_SVHN, val_dataset_SVHN = random_split(traindata_SVHN, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_SVHN = torch.utils.data.DataLoader(traindata_SVHN, batch_size=32, shuffle=True)
valload_SVHN = torch.utils.data.DataLoader(val_dataset_SVHN, batch_size=32, shuffle=False)
testload_SVHN = torch.utils.data.DataLoader(testdata_SVHN, batch_size=32, shuffle=False)

In [25]:
#Test MNIST model on SVHN dataset

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

wandb.finish()


Test loss: 2.335
Test accuracy: 19.50%


In the third step you are performing transfer learning from MNIST to SVHN.

In [26]:
early_stop_patience = 6
early_stopping_counter = 0
epochs = 500

In [27]:
#Transfer learning from MNIST to SVHN with feature extraction
wandb.init(project="DeepLearning", name="MNIST Transfer learning to SVHN - Feature extraction")

model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)

#Freeze model weights
for param in model.parameters():
    param.requires_grad = False

model.classifier[4] = nn.Linear(128, 10)
model = model.to(device)
optimizer = torch.optim.SGD(model.classifier[4].parameters(), lr=0.0001)

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_SVHN)
    wandb.log({
        "Training loss - Transfer Learning - Feature extraction, MNIST MODEL on SVHN Dataset": average_loss
        })

    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_SVHN:
            images = images.to(device)
            if images.shape[1] == 3:  # RGB
                images = images.mean(dim=1, keepdim=True)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_SVHN)

        wandb.log({
            "Validation loss - Transfer Learning - Feature extraction, MNIST MODEL on SVHN Dataset": average_val_loss
        })

    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_SVHN_Feature_Extraction.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter: {early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 2.336, val loss: 2.257, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 2.266, val loss: 2.231, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 2.242, val loss: 2.210, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 2.225, val loss: 2.193, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 2.209, val loss: 2.177, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 2.194, val loss: 2.164, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 2.180, val loss: 2.151, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 2.171, val loss: 2.141, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 2.161, val loss: 2.133, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 2.150, val loss: 2.124, lr: 0.000100


In [28]:
#Test transfer model on SVHN dataset with feature extraction

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_SVHN_Feature_Extraction.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

wandb.finish()

Test loss: 2.113
Test accuracy: 27.12%


"Training loss - Transfer Learning - Feature extraction, MNIST MODEL on SVHN Dataset",█▅▄▄▃▃▂▂▁▁
"Validation loss - Transfer Learning - Feature extraction, MNIST MODEL on SVHN Dataset",█▇▆▅▄▃▂▂▁▁
"Training loss - Transfer Learning - Feature extraction, MNIST MODEL on SVHN Dataset",2.14965
"Validation loss - Transfer Learning - Feature extraction, MNIST MODEL on SVHN Dataset",2.12426


In [29]:
early_stop_patience = 6
early_stopping_counter = 0
epochs = 500

In [30]:
# Now transfer learning with fine-tuning:
wandb.init(project="DeepLearning", name="MNIST Transfer learning to SVHN - Fine tuning")

model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)


for param in model.parameters():
    param.requires_grad = True

model.classifier[4] = nn.Linear(128, 10)
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_SVHN)
    wandb.log({
        "Training loss - Transfer Learning - Fine tuning, MNIST MODEL on SVHN Dataset": average_loss
        })

    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_SVHN:
            images = images.to(device)
            if images.shape[1] == 3:  # RGB
                images = images.mean(dim=1, keepdim=True)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_SVHN)

        wandb.log({
            "Validation loss - Transfer Learning - Fine tuning MNIST MODEL on SVHN Dataset": average_val_loss
        })

    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_SVHN_Fine_Tuning.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement, increasing counter: {early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break
        
    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 2.194, val loss: 2.084, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 2.023, val loss: 1.925, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 1.862, val loss: 1.749, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 1.706, val loss: 1.604, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 1.562, val loss: 1.475, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 1.438, val loss: 1.346, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 1.330, val loss: 1.254, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 1.239, val loss: 1.174, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 1.160, val loss: 1.105, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 1.094, val loss: 1.043, lr: 0.000100


In [31]:
#Test transfer model on SVHN dataset with fine tuning

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_SVHN_Fine_Tuning.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

wandb.finish()

Test loss: 1.060
Test accuracy: 71.80%


"Training loss - Transfer Learning - Fine tuning, MNIST MODEL on SVHN Dataset",█▇▆▅▄▃▃▂▁▁
Validation loss - Transfer Learning - Fine tuning MNIST MODEL on SVHN Dataset,█▇▆▅▄▃▂▂▁▁
"Training loss - Transfer Learning - Fine tuning, MNIST MODEL on SVHN Dataset",1.09398
Validation loss - Transfer Learning - Fine tuning MNIST MODEL on SVHN Dataset,1.04342
